# Импорты

In [1]:
!pip install catboost
!pip install pymorphy3
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 83.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 5.4 MB/s eta 0:00:00


In [17]:
import pandas as pd
import numpy as np
import re
from scipy.sparse import hstack
import optuna

import pymorphy3
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    confusion_matrix,
    mean_absolute_error,
    root_mean_squared_error,
)

import sys
import subprocess
from mord import LogisticAT, LogisticIT, OrdinalRidge


In [3]:
from nltk.corpus import stopwords
import nltk

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

## Загрузка и исследование данных

In [4]:
df = pd.read_excel('nerd_analytics_v25.xlsx', sheet_name='reviews')

In [5]:
df = df[['id', 'comment', 'sentiment', 'rating', 'keywords_positive', 'keywords_neutral', 'keywords_negative', 'product', 'final_category']]

In [6]:
df.shape[0]

1235

In [7]:
df[['id', 'comment', 'sentiment', 'keywords_positive', 'keywords_neutral', 'keywords_negative', 'product', 'final_category']].describe()

,id,comment,sentiment,keywords_positive,keywords_neutral,keywords_negative,product,final_category
count,1235,1235,1235,815,397,189,1235,1235
unique,1235,65,3,29,12,16,6,5
top,266a542e-a9f3-4127-916e-76e017b10e1b,"Ответили быстро, не пришлось долго ждать",positive,приятно,целом,долго,аналитический модуль,качество ответа
freq,1,45,770,59,66,69,249,286


In [8]:
df.describe()

,rating
count,1235.000000
mean,3.710121
std,1.155714
min,1.000000
25%,3.000000
50%,4.000000
75%,5.000000
max,5.000000


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1235 entries, 0 to 1234
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id                 1235 non-null   object
 1   comment            1235 non-null   object
 2   sentiment          1235 non-null   object
 3   rating             1235 non-null   int64 
 4   keywords_positive  815 non-null    object
 5   keywords_neutral   397 non-null    object
 6   keywords_negative  189 non-null    object
 7   product            1235 non-null   object
 8   final_category     1235 non-null   object
dtypes: int64(1), object(8)
memory usage: 87.0+ KB


In [10]:
df['product'].value_counts()

,count
product,
аналитический модуль,249
платёжный сервис,210
API интеграция,206
мобильное приложение,204
веб-сервис,193
личный кабинет,173


In [11]:
df.isna().sum(), df.duplicated().sum()

(id                      0
 comment                 0
 sentiment               0
 rating                  0
 keywords_positive     420
 keywords_neutral      838
 keywords_negative    1046
 product                 0
 final_category          0
 dtype: int64,
 np.int64(0))

# Определение рейтинга отзыва

## Подготовка данных

In [12]:
df['rating'].value_counts().sort_index()

,count
rating,
1,66
2,130
3,269
4,401
5,369


In [13]:
df1 = df.copy()

In [14]:
morph = pymorphy3.MorphAnalyzer()
russian_stopwords = stopwords.words('russian')
bad_stopwords = ['не', 'нет', 'ни', 'без']

russian_stopwords = [
    word for word in russian_stopwords
    if word not in bad_stopwords
]

def clean_text(text):

    text = text.lower()
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"www\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"[^а-яa-zё\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    words = text.split()
    lemmas = []

    for word in words:
        if word not in russian_stopwords:
            lemma = morph.parse(word)[0].normal_form
            lemmas.append(lemma)

    return " ".join(lemmas)

df1['clean_comment'] = df1['comment'].apply(clean_text)

In [15]:
RANDOM_STATE = 42

## Перебор моделей и гиперпараметров

In [18]:
rate_df = df1[
    [
        'clean_comment',
        'product',
        'final_category',
        'sentiment',
        'rating',
    ]
].copy()

X_all = rate_df.drop(columns=['rating'])
y_all = rate_df['rating'].astype(int)
groups_all = rate_df['clean_comment']

N_SPLITS = 5
CV_RANDOM_STATES = [RANDOM_STATE, 101, 202, 303, 404]

In [30]:
def build_sparse_features(train_part, valid_part):
    word_vectorizer = TfidfVectorizer(max_features=25000, ngram_range=(1, 2), min_df=2, max_df=0.95)
    char_vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), max_features=15000)
    cat_cols = ['product', 'final_category', 'sentiment']
    try:
        encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
    except TypeError:
        encoder = OneHotEncoder(handle_unknown='ignore', sparse=True)

    X_train_word = word_vectorizer.fit_transform(train_part['clean_comment'])
    X_valid_word = word_vectorizer.transform(valid_part['clean_comment'])
    X_train_char = char_vectorizer.fit_transform(train_part['clean_comment'])
    X_valid_char = char_vectorizer.transform(valid_part['clean_comment'])
    X_train_cat = encoder.fit_transform(train_part[cat_cols].fillna('').astype(str))
    X_valid_cat = encoder.transform(valid_part[cat_cols].fillna('').astype(str))

    return hstack([X_train_word, X_train_char, X_train_cat]), hstack([X_valid_word, X_valid_char, X_valid_cat])


def build_dense_from_sparse(X_train_sparse, X_valid_sparse, n_components):
    n_components = min(n_components, X_train_sparse.shape[1] - 1)
    svd = TruncatedSVD(n_components=n_components, random_state=RANDOM_STATE)
    X_train_dense = svd.fit_transform(X_train_sparse)
    X_valid_dense = svd.transform(X_valid_sparse)
    return X_train_dense, X_valid_dense


def build_model_and_params(trial):
    model_type = trial.suggest_categorical('model_type', ['LogisticAT', 'LogisticIT', 'OrdinalRidge', 'CatBoost'])

    if model_type in ['LogisticAT', 'LogisticIT', 'OrdinalRidge']:
        params = {
            'alpha': trial.suggest_float('ord_alpha', 0.05, 20.0, log=True),
            'n_components': trial.suggest_int('ord_n_components', 100, 260),
        }
        return model_type, params

    params = {
        'iterations': trial.suggest_int('cb_iterations', 200, 900),
        'depth': trial.suggest_int('cb_depth', 4, 10),
        'learning_rate': trial.suggest_float('cb_learning_rate', 0.01, 0.2, log=True),
        'l2_leaf_reg': trial.suggest_int('cb_l2_leaf_reg', 1, 12),
        'random_strength': trial.suggest_float('cb_random_strength', 1.0, 10.0),
        'bagging_temperature': trial.suggest_float('cb_bagging_temperature', 0.0, 10.0),
        'border_count': trial.suggest_int('cb_border_count', 64, 255),
        'auto_class_weights': trial.suggest_categorical('cb_auto_class_weights', ['Balanced', None]),
    }
    return model_type, params


def make_predictions(model_type, model_params, X_tr_sparse, X_val_sparse, y_tr, y_val=None):
    if model_type == 'CatBoost':
        model = CatBoostClassifier(
            **model_params,
            loss_function='MultiClass',
            eval_metric='MultiClass',
            random_state=RANDOM_STATE,
            task_type='GPU',
            verbose=False,
        )
        if y_val is None:
            model.fit(X_tr_sparse, y_tr, verbose=False)
        else:
            model.fit(X_tr_sparse, y_tr, eval_set=(X_val_sparse, y_val), early_stopping_rounds=80, verbose=False)
        pred = model.predict(X_val_sparse).flatten()
        return np.clip(np.rint(pred).astype(int), 1, 5)

    X_tr_dense, X_val_dense = build_dense_from_sparse(
        X_tr_sparse,
        X_val_sparse,
        n_components=model_params['n_components'],
    )

    if model_type == 'LogisticAT':
        model = LogisticAT(alpha=model_params['alpha'])
    elif model_type == 'LogisticIT':
        model = LogisticIT(alpha=model_params['alpha'])
    else:
        model = OrdinalRidge(alpha=model_params['alpha'])

    model.fit(X_tr_dense, y_tr)
    pred = model.predict(X_val_dense)
    return np.clip(np.rint(pred).astype(int), 1, 5)


def get_cv_predictions(model_type, model_params, data, cv_random_states):
    cv_results = []
    y_true_all = []
    y_pred_all = []

    for cv_seed in cv_random_states:
        cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=cv_seed)

        for fold, (tr_fold_idx, val_fold_idx) in enumerate(
            cv.split(data.drop(columns=['rating']), data['rating'], groups=data['clean_comment']),
            start=1,
        ):
            tr_part = data.iloc[tr_fold_idx]
            val_part = data.iloc[val_fold_idx]

            X_tr_sparse, X_val_sparse = build_sparse_features(
                tr_part.drop(columns=['rating']),
                val_part.drop(columns=['rating']),
            )
            y_tr = tr_part['rating'].astype(int).values
            y_val = val_part['rating'].astype(int).values
            pred = make_predictions(model_type, model_params, X_tr_sparse, X_val_sparse, y_tr, y_val)

            cv_results.append({
                'seed': cv_seed,
                'fold': fold,
                'mae': mean_absolute_error(y_val, pred),
                'rmse': root_mean_squared_error(y_val, pred),
                'accuracy': (pred == y_val).mean(),
                'within_1': (np.abs(pred - y_val) <= 1).mean(),
            })
            y_true_all.extend(y_val)
            y_pred_all.extend(pred)

    return pd.DataFrame(cv_results), np.array(y_true_all), np.array(y_pred_all)


def objective(trial):
    model_type, model_params = build_model_and_params(trial)
    cv_results, _, _ = get_cv_predictions(model_type, model_params, rate_df, CV_RANDOM_STATES)
    return float(cv_results['mae'].mean())

In [20]:
sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
study = optuna.create_study(direction='minimize', sampler=sampler)
study.optimize(objective, n_trials=50)

print('BEST MEAN CV MAE:')
print(study.best_value)
print('BEST PARAMS:')
print(study.best_params)

[I 2026-05-25 16:49:18,681] A new study created in memory with name: no-name-20d07286-a69e-4818-9679-c2a22ce7e164
[I 2026-05-25 16:49:32,258] Trial 0 finished with value: 0.5155515263964494 and parameters: {'model_type': 'LogisticIT', 'ord_alpha': 0.12733267608983528, 'ord_n_components': 125}. Best is trial 0 with value: 0.5155515263964494.
[I 2026-05-25 16:49:54,990] Trial 1 finished with value: 0.5146575270000355 and parameters: {'model_type': 'LogisticIT', 'ord_alpha': 0.05656295542959041, 'ord_n_components': 256}. Best is trial 1 with value: 0.5146575270000355.
[I 2026-05-25 16:50:12,071] Trial 2 finished with value: 0.5131780496350188 and parameters: {'model_type': 'LogisticAT', 'ord_alpha': 0.30947571316582956, 'ord_n_components': 184}. Best is trial 2 with value: 0.5131780496350188.
[I 2026-05-25 16:50:25,132] Trial 3 finished with value: 0.5298765054062266 and parameters: {'model_type': 'OrdinalRidge', 'ord_alpha': 0.2878378525926593, 'ord_n_components': 158}. Best is trial 2 w

BEST MEAN CV MAE:
0.3436913179453137
BEST PARAMS:
{'model_type': 'CatBoost', 'cb_iterations': 436, 'cb_depth': 4, 'cb_learning_rate': 0.08512825639759057, 'cb_l2_leaf_reg': 9, 'cb_random_strength': 3.630280534343451, 'cb_bagging_temperature': 3.8388575694499907, 'cb_border_count': 100, 'cb_auto_class_weights': None}


## Обучение лучшей модели

In [31]:
best_params = dict(study.best_params)
best_model_type = best_params.pop('model_type')

if best_model_type == 'CatBoost':
    model_params = {
        'iterations': best_params['cb_iterations'],
        'depth': best_params['cb_depth'],
        'learning_rate': best_params['cb_learning_rate'],
        'l2_leaf_reg': best_params['cb_l2_leaf_reg'],
        'random_strength': best_params['cb_random_strength'],
        'bagging_temperature': best_params['cb_bagging_temperature'],
        'border_count': best_params['cb_border_count'],
        'auto_class_weights': best_params['cb_auto_class_weights'],
    }
else:
    model_params = {
        'alpha': best_params['ord_alpha'],
        'n_components': best_params['ord_n_components'],
    }

cv_results_df, y_true_cv, y_pred_cv = get_cv_predictions(best_model_type, model_params, rate_df, CV_RANDOM_STATES)

print(f'\nBest model type: {best_model_type}')
print(f'CV seeds: {CV_RANDOM_STATES}')
print(f"MAE: {cv_results_df['mae'].mean():.4f} +/- {cv_results_df['mae'].std():.4f}")
print(f"RMSE: {cv_results_df['rmse'].mean():.4f} +/- {cv_results_df['rmse'].std():.4f}")
print(f"Accuracy (exact): {cv_results_df['accuracy'].mean():.4f} +/- {cv_results_df['accuracy'].std():.4f}")
print(f"Accuracy (within 1): {cv_results_df['within_1'].mean():.4f} +/- {cv_results_df['within_1'].std():.4f}")
print('\nFOLD METRICS SUMMARY:\n')
print(cv_results_df[['mae', 'rmse', 'accuracy', 'within_1']].describe())

X_full_sparse, _ = build_sparse_features(
    rate_df.drop(columns=['rating']),
    rate_df.drop(columns=['rating']),
)
y_full = rate_df['rating'].astype(int).values

if best_model_type == 'CatBoost':
    final_model = CatBoostClassifier(
        **model_params,
        loss_function='MultiClass',
        eval_metric='MultiClass',
        random_state=RANDOM_STATE,
        task_type='GPU',
        verbose=False,
    )
    final_model.fit(X_full_sparse, y_full, verbose=False)
    final_model.save_model('reviews_ratings.cbm')
else:
    import joblib

    X_full_dense, _ = build_dense_from_sparse(
        X_full_sparse,
        X_full_sparse,
        n_components=model_params['n_components'],
    )
    if best_model_type == 'LogisticAT':
        final_model = LogisticAT(alpha=model_params['alpha'])
    elif best_model_type == 'LogisticIT':
        final_model = LogisticIT(alpha=model_params['alpha'])
    else:
        final_model = OrdinalRidge(alpha=model_params['alpha'])

    final_model.fit(X_full_dense, y_full)
    joblib.dump(final_model, 'reviews_ratings.pkl')


Best model type: CatBoost
CV seeds: [42, 101, 202, 303, 404]
MAE: 0.3437 +/- 0.1390
RMSE: 0.5734 +/- 0.1246
Accuracy (exact): 0.6563 +/- 0.1390
Accuracy (within 1): 1.0000 +/- 0.0000

FOLD METRICS SUMMARY:

             mae       rmse   accuracy  within_1
count  25.000000  25.000000  25.000000      25.0
mean    0.343691   0.573394   0.656309       1.0
std     0.139001   0.124626   0.139001       0.0
min     0.109827   0.331401   0.388664       1.0
25%     0.250996   0.500995   0.572368       1.0
50%     0.342629   0.585346   0.657371       1.0
75%     0.427632   0.653935   0.749004       1.0
max     0.611336   0.781880   0.890173       1.0


In [32]:
labels = [1, 2, 3, 4, 5]
cm = confusion_matrix(y_true_cv, y_pred_cv, labels=labels)
cm_df = pd.DataFrame(cm, index=[f'true_{x}' for x in labels], columns=[f'pred_{x}' for x in labels])
print('\nCONFUSION MATRIX OVER ALL CV PREDICTIONS:\n')
print(cm_df)


CONFUSION MATRIX OVER ALL CV PREDICTIONS:

        pred_1  pred_2  pred_3  pred_4  pred_5
true_1      46     284       0       0       0
true_2     179     471       0       0       0
true_3       0       0    1345       0       0
true_4       0       0       0    1049     956
true_5       0       0       0     740    1105
